# PHA–enzyme distances and contacts

Which enzyme residues remain close to the **whole PHA molecule** in an existing trajectory? This notebook calls `iphasimulator.analysis_contacts`; all reusable calculations live in the package.

Each value is the minimum atom-pair distance between one enzyme residue and **all selected PHA heavy atoms**, using the periodic box from the same frame. The modes are all enzyme heavy atoms and side-chain heavy atoms. These are proximity/contact measurements, not binding affinity, evidence of catalytic activity, or a convergence assessment.

Use the `ipha_clean` Python environment on this machine. Other environments need `python -m pip install -e ".[analysis]"` from the project root, plus a Jupyter kernel.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

# Works when Jupyter starts in either the project root or this notebook folder.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/iphasimulator/analysis_contacts.py").is_file())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from iphasimulator.analysis_contacts import (
    load_config, open_universe, select_atoms, run_contacts,
    load_distances, summarise, plot_contacts,
)

CONFIG_PATH = ROOT / "examples/enzyme_contacts_GK13_P3HO_4.yaml"
config = load_config(CONFIG_PATH)
print("Python:", sys.executable)
print("Configuration:", CONFIG_PATH)

## Configure the window and sampling

The supplied YAML points to `GK13_P3HO_4_gromacs/step7_production.tpr` and **only** `production_combined_1us.xtc`. Do not append source continuation files or use a fitted trajectory. The actual verified time range is **0–1480.5 ns**, with **14,806 frames** at **0.1 ns** intervals; its filename does not define its duration.

The initial window is explicitly the entire available trajectory. `END_NS = None` means its actual final timestamp; no equilibration period is silently discarded. Boundaries are inclusive. `SAMPLE_INTERVAL_NS = None` uses every stored frame. An explicit interval must be an integer multiple of the native interval and starts at the first stored frame inside your window.

**To run the full analysis, change `PREVIEW_FRAMES` below to `None`, then run this cell and the following cells.** With the initial sampling setting, that calculates both modes for all 14,806 frames. A preview instead spreads 31 samples over the same window. Every run creates a new directory under `examples/output/enzyme_contacts/`.

In [ ]:
START_NS = 0.0
END_NS = None
SAMPLE_INTERVAL_NS = None
PREVIEW_FRAMES = 31  # Set to None for the full analysis.
MODES = ("all_heavy", "sidechain_heavy")
PRIMARY_CUTOFF_A = 4.5

config = replace(
    config, start_ns=START_NS, end_ns=END_NS,
    sample_interval_ns=SAMPLE_INTERVAL_NS, preview_frames=PREVIEW_FRAMES,
    modes=MODES, primary_cutoff_A=PRIMARY_CUTOFF_A,
)
config.validate()
display(config)

## Check selections from the topology

The verified protein selection has 245 residues (`LEU1` through `PRO245`, segment `seg_0_PROA`), 3,701 atoms, and 1,898 heavy atoms. The PHA is one `LIG` residue: topology residue index 245, residue number 246, segment `seg_1_LIG`, 99 total atoms, **41 heavy atoms (32 C + 9 O)**. The accompanying `topol.top` includes one `LIG` molecule defined in `toppar/LIG.itp`.

Side-chain mode selects 917 protein atoms. It excludes N, CA, C, O and terminal oxygen aliases, including the actual `OT1`/`OT2` names in this topology. All 25 glycines keep their rows as **NaN/unavailable**. Atom indices stay filtered throughout the calculations; accessing residue metadata never expands the distance selections.

MDAnalysis verifies the topology/trajectory atom counts. XTC stores coordinates without atom identities, so atom-order compatibility also relies on using the matching original system topology. The completed preview independently checks sampled distances and bonded geometry; see `docs/enzyme_contacts.md`.

In [ ]:
with open_universe(config.topology, config.trajectory) as universe:
    selections = select_atoms(universe, config.protein_selection, config.pha_selection)
    print("Topology atoms:", universe.atoms.n_atoms)
    print("Trajectory atoms:", universe.trajectory.n_atoms)
    print("Stored frames:", len(universe.trajectory))
    print("Protein atoms:", len(selections.protein))
    print("PHA heavy atoms:", len(selections.pha))
    print("PHA residues:", selections.pha.residues)
    display(pd.DataFrame(selections.residues).assign(
        all_heavy_atoms=selections.counts["all_heavy"],
        sidechain_heavy_atoms=selections.counts["sidechain_heavy"],
    ))

## Run and inspect provenance

The workflow first streams **every stored frame** to validate strictly increasing timestamps, sampling, atom counts, and finite nondegenerate periodic boxes. It then calculates only the requested frames, without fitting or loading a coordinate trajectory into memory. The first pass therefore reads the whole XTC even for a preview.

Occupancy is **100 × number of analysed frames with minimum distance < cutoff / number of analysed frames**. Each residue contributes at most one event per frame. The initial cutoffs are 4.0, 4.5 and 5.0 Å. Unavailable rows remain NaN in the matrices, CSV and figures. Preview spacing may alternate because samples must fall on stored frames; actual timestamps and intervals are saved.

These are sampled-frame occupancies: correlated samples are not independent observations, and percentages are not uncertainty estimates.

In [ ]:
RUN_DIR = run_contacts(config)
print("Saved run:", RUN_DIR)
provenance = json.loads((RUN_DIR / "provenance.json").read_text())
display(provenance["inspection"])
display(provenance["sampling"])

In [ ]:
summary = pd.read_csv(RUN_DIR / "residue_summary.csv")
# Highest preview/full sampled occupancies per mode; preserve the identity columns.
occupancy_column = f"occupancy_lt_{config.primary_cutoff_A:g}_A_pct"
for mode in config.modes:
    print(mode)
    display(summary[summary["mode"] == mode].sort_values(
        occupancy_column, ascending=False).head(15))

for mode in config.modes:
    display(Image(filename=str(RUN_DIR / f"distance_heatmap_{mode}.png")))
    display(Image(filename=str(RUN_DIR / f"contact_occupancy_{mode}.png")))

## Reuse the saved matrices

`distances.npz` stores residue × sampled-frame matrices in Å, times in ns, original frame indices, same-frame boxes, residue identifiers, selected atom counts and provenance. It loads with `allow_pickle=False`. The CSV has one row per residue and mode, including selected atom count, analysed/valid frame counts, mean/median distance and occupancy at every cutoff.

Run the next cell independently after assigning an existing `RUN_DIR` to change cutoffs without reading the trajectory. The last cell is optional: enable it to write a new set of plots with a more focused distance colour scale. Values above that scale are clipped only visually; stored distances stay unchanged. Grey marks missing rows. Heatmap columns represent samples using midpoint time bins, with no interpolation of the distance values.

In [ ]:
# To inspect an existing run after restarting the kernel, set RUN_DIR explicitly:
# RUN_DIR = ROOT / "examples/output/enzyme_contacts/<your-run-directory>"
saved = load_distances(RUN_DIR / "distances.npz")
NEW_CUTOFFS_A = (4.0, 4.5, 5.0, 6.0)
revised_summary = summarise(saved, NEW_CUTOFFS_A)
display(revised_summary.head())

In [ ]:
WRITE_REVISED_PLOTS = False
if WRITE_REVISED_PLOTS:
    from datetime import datetime, timezone
    derived_dir = RUN_DIR / ("replot_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    derived_dir.mkdir(exist_ok=False)
    revised_summary.to_csv(derived_dir / "residue_summary.csv", index=False, na_rep="NaN")
    plot_contacts(saved, derived_dir, NEW_CUTOFFS_A, primary_cutoff_A=4.5, distance_vmax_A=15)
    print("Revised figures:", derived_dir)

## Terminal equivalent and scope

From the project root, using the verified local interpreter:

```bash
/opt/homebrew/Caskroom/miniconda/base/envs/ipha_clean/bin/python examples/run_enzyme_contacts.py examples/enzyme_contacts_GK13_P3HO_4.yaml --full
```

Omit `--full` for the default preview. For an explicit alternative window/sampling, append, for example, `--start-ns 100 --end-ns 1000 --sample-interval-ns 1.0`. Choose such windows deliberately; this example does not assert equilibration at 100 ns. See `docs/enzyme_contacts.md` for file formats, validation and implementation details.

Catalytic-residue annotations, repeat-unit mapping, APO RMSF, catalytic geometry and batch comparisons are later extensions. The present results measure proximity to the whole PHA.